# Area-recall curves

This notebook produces cumulative area-recall curves for the global and regional classifiers from `02a-create_probability_maps.ipynb`. Paths follow [`PathConfigManager`](lib/paths.py).

## Notebook setup

These cells set paths and run parameters from the selected config file.

### Config

In [ ]:
config_file = "config/.run_config.yml"

In [ ]:
from lib.paths import PathConfigManager

pcm = PathConfigManager(config_file, notebook="02")

# =====================
# Filestructure
# =====================

grid_data_filepath = pcm.GRID_DATA_PATH
training_filepath = pcm.TRAINING_DATA_PATH
output_dir = pcm.OUTPUT_DIR

pcm.create_directories()
pcm.AREA_RECALL_DIR.mkdir(parents=True, exist_ok=True)


# =====================
# Notebook scope
# =====================

use_extracted_data = pcm.use_extracted_data

create_SVM = False

# =====================
# Run parameters
# =====================

n_jobs = pcm.config["n_jobs"]
overwrite = pcm.config["overwrite_output"]
verbose = pcm.config["verbose"]

### Imports

In [16]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cmcrameri.cm as cmc
from pathlib import Path

plt.style.use(pcm.CONFIG_DIR / "thesis.mplstyle")
plt.rcParams["axes.prop_cycle"] = mpl.cycler(
    color=[cmc.batlow(i / 7) for i in range(8)]
)

TW = 150 / 25.4

### Input and output files

In [ ]:
training_data = pd.read_csv(pcm.TRAINING_DATA_PATH)
deposits = (
    training_data[training_data["label"] == "positive"][["lon", "lat", "age (Ma)", "region"]]
    .drop_duplicates()
)

regions = training_data["region"].unique().tolist()
algorithms = ("PU", "SVM") if create_SVM else ("PU",)
grid_resolution = pcm.config.get("grid_resolution", 1.0)

## Area-recall curves

Each plot shows what fraction of known copper deposits (recall) is captured when selecting the top X% of grid cells ranked by predicted probability. A perfect classifier stays near recall = 1 for small area fractions; a random classifier follows the diagonal.

One plot is produced per classifier (global and each regional), plus a combined comparison at the end.

In [19]:
def compute_area_recall(probabilities_df, deposits_df, grid_res):
    """
    Returns (area_fractions, recall_fractions) sorted by descending probability.
    Deposits are snapped to the nearest grid node. Unmatched deposits are dropped
    with a warning — this should not occur given a consistent grid resolution.
    """
    deps = deposits_df.copy()
    deps["_lon_snap"] = (deps["lon"] / grid_res).round() * grid_res
    deps["_lat_snap"] = (deps["lat"] / grid_res).round() * grid_res

    prob = probabilities_df.copy()
    prob["_lon_snap"] = (prob["lon"] / grid_res).round() * grid_res
    prob["_lat_snap"] = (prob["lat"] / grid_res).round() * grid_res

    deposit_keys = deps[["_lon_snap", "_lat_snap", "age (Ma)"]].drop_duplicates()
    deposit_keys = deposit_keys.assign(_is_deposit=1)

    df = prob.merge(deposit_keys, on=["_lon_snap", "_lat_snap", "age (Ma)"], how="left")
    df["_is_deposit"] = df["_is_deposit"].fillna(0).astype(int)

    deposit_coords = deposit_keys[["_lon_snap", "_lat_snap", "age (Ma)"]].copy()
    matched_coords = (
        df.loc[df["_is_deposit"] == 1, ["_lon_snap", "_lat_snap", "age (Ma)"]]
        .drop_duplicates()
    )
    unmatched = deposit_coords.merge(
        matched_coords, on=["_lon_snap", "_lat_snap", "age (Ma)"], how="left", indicator=True
    )
    unmatched = unmatched[unmatched["_merge"] == "left_only"]
    if len(unmatched) > 0:
        print(f"Warning: {len(unmatched)} deposit(s) had no matching grid point and were dropped:")
        for _, row in unmatched.iterrows():
            print(f"  lon={row['_lon_snap']:.2f}, lat={row['_lat_snap']:.2f}, age={row['age (Ma)']}")

    df = df.sort_values("probability", ascending=False).reset_index(drop=True)
    n_total = len(df)
    n_deposits = df["_is_deposit"].sum()
    if n_deposits == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 0.0])

    area = np.arange(1, n_total + 1) / n_total
    recall = df["_is_deposit"].cumsum().values / n_deposits
    return area, recall

In [ ]:
plot_data = {}

for algorithm in algorithms:
    figures_dir = pcm.AREA_RECALL_DIR

    prob_file = pcm.GRID_PROBABILITIES_PATH
    if not prob_file.is_file():
        if verbose:
            print(f"Skipping {algorithm} global: {prob_file} not found")
        continue

    prob_global = pd.read_csv(prob_file)
    area, recall = compute_area_recall(prob_global, deposits, grid_resolution)
    label = f"{algorithm} global"
    plot_data[label] = {"area": area, "recall_global": recall}

    with mpl.rc_context({"axes.grid": True, "axes.grid.axis": "y"}):
        fig, ax = plt.subplots(figsize=(TW, TW * 0.62))
        ax.plot(area, recall, label="Global recall")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
        ax.set_xlabel("Area fraction")
        ax.set_ylabel("Recall")
        ax.set_title(f"Area-recall: {algorithm} global classifier")
        ax.legend()
        for ext in (".pdf", ".png"):
            fig.savefig(f"{figures_dir / 'global'}{ext}")
        plt.show()

    for region in regions:
        region_code = "_".join(region.lower().split())
        prob_file = pcm.GRID_PROBABILITIES_PATH.with_name(f"grid_probabilities_{region_code}.csv")
        if not prob_file.is_file():
            if verbose:
                print(f"Skipping {algorithm} {region}: {prob_file} not found")
            continue

        prob_regional = pd.read_csv(prob_file)

        area_g, recall_g = compute_area_recall(prob_regional, deposits, grid_resolution)
        area_r, recall_r = compute_area_recall(
            prob_regional[prob_regional["region"] == region],
            deposits[deposits["region"] == region],
            grid_resolution,
        )
        label = f"{algorithm} {region}"
        plot_data[label] = {
            "area_global": area_g,
            "recall_global": recall_g,
            "area_regional": area_r,
            "recall_regional": recall_r,
        }

        with mpl.rc_context({"axes.grid": True, "axes.grid.axis": "y"}):
            fig, ax = plt.subplots(figsize=(TW, TW * 0.62))
            ax.plot(area_g, recall_g, label="All deposits (global recall)")
            ax.plot(area_r, recall_r, label=f"{region} deposits (regional recall)")
            ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
            ax.set_xlabel("Area fraction")
            ax.set_ylabel("Recall")
            ax.set_title(f"Area-recall: {algorithm} — {region}")
            ax.legend()
            for ext in (".pdf", ".png"):
                fig.savefig(f"{figures_dir / region_code}{ext}")
            plt.show()

In [ ]:
for algorithm in algorithms:
    figures_dir = pcm.AREA_RECALL_DIR
    algo_entries = {k: v for k, v in plot_data.items() if k.startswith(algorithm)}
    if not algo_entries:
        continue

    with mpl.rc_context({"axes.grid": True, "axes.grid.axis": "y"}):
        fig, ax = plt.subplots(figsize=(TW, TW * 0.62))
        for label, data in algo_entries.items():
            x = data.get("area_global", data.get("area"))
            ax.plot(x, data["recall_global"], label=label.replace(f"{algorithm} ", ""))
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
        ax.set_xlabel("Area fraction")
        ax.set_ylabel("Recall (all deposits)")
        ax.set_title(f"Area-recall: {algorithm} — all classifiers")
        ax.legend()
        for ext in (".pdf", ".png"):
            fig.savefig(f"{figures_dir / 'combined'}{ext}")
        plt.show()

In [ ]:
rows = []
for label, data in plot_data.items():
    classifier = label.replace("PU ", "")
    if "area_regional" not in data:  # global classifier — keys: "area", "recall_global"
        for a, r in zip(data["area"], data["recall_global"]):
            rows.append({"classifier": classifier, "curve_type": "global", "area": a, "recall": r})
    else:  # regional classifier — keys: "area_global", "recall_global", "area_regional", "recall_regional"
        for a, r in zip(data["area_global"], data["recall_global"]):
            rows.append({"classifier": classifier, "curve_type": "global", "area": a, "recall": r})
        for a, r in zip(data["area_regional"], data["recall_regional"]):
            rows.append({"classifier": classifier, "curve_type": "regional", "area": a, "recall": r})
pd.DataFrame(rows).to_csv(pcm.AREA_RECALL_DATA_PATH, index=False)